# 03 Population Definition and Harmonization

Fase 3: definicion de poblacion LGBT+, estandarizacion de estado y manejo de codigos especiales.

In [1]:
from pathlib import Path
import sys
import polars as pl
root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
endiseg_path = root / 'source_inegi' / 'conjunto_de_datos_tmodulo_endiseg_2021' / 'conjunto_de_datos' / 'conjunto_de_datos_tmodulo_endiseg_2021.csv'
df = pl.read_csv(endiseg_path, infer_schema_length=10000, ignore_errors=True).rename(lambda c: c.lower())
df.select(['ent', 'factor', 'p8_1', 'p8_1a', 'p9_1', 'filtro_10_5']).head()

ent,factor,p8_1,p8_1a,p9_1,filtro_10_5
i64,i64,i64,str,i64,i64
1,336,5,"""""",1,2
1,633,4,"""""",2,2
1,1267,4,"""""",2,2
1,633,4,"""""",2,2
1,240,4,"""""",2,2


In [2]:
num = lambda c: pl.col(c).cast(pl.Float64, strict=False)
analytic = df.with_columns([
    pl.col('ent').cast(pl.Utf8).str.zfill(2).alias('state_code'),
    num('factor').alias('weight'),
    ((num('p8_1').is_in([2,3,4,5,6])) | (num('p8_1a').is_in([2,3,4])) | (num('p9_1').is_in([2,3,4,5])) | (num('filtro_10_5') == 1)).alias('is_lgbt')
])
analytic.select(['state_code','weight','is_lgbt']).head()

state_code,weight,is_lgbt
str,f64,bool
"""01""",336.0,true
"""01""",633.0,true
"""01""",1267.0,true
"""01""",633.0,true
"""01""",240.0,true


Notas de armonizacion:
- `state_code` se normaliza a 2 digitos.
- `weight` se toma de `factor`.
- Codigos 9/98/99 se tratan como no respuesta en indicadores especificos.